# 04 — Time Series Deep Learning
## Wind Turbine Gearbox Anomaly Detection

Bu notebook, zaman serisi anomali tespiti için derin öğrenme mimarilerini uygular.

**Modeller:**
1. **LSTM** — Long Short-Term Memory (sequence classification)
2. **TCN** — Temporal Convolutional Network (dilated causal convolutions)
3. **Transformer Encoder** — Self-attention tabanlı (PatchTST benzeri)

**Teknik:**
- Sliding window yaklaşımı (window_size=48, stride=1)
- Early stopping + learning rate scheduler
- F1, ROC-AUC, PR-AUC metrikleri

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
import glob
import warnings
warnings.filterwarnings('ignore')

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    f1_score, roc_auc_score, average_precision_score,
    precision_score, recall_score, roc_curve, precision_recall_curve
)

tf.random.set_seed(42)
np.random.seed(42)
plt.rcParams['figure.figsize'] = (12, 6)
sns.set_style('whitegrid')
print(f'TensorFlow: {tf.__version__}')
print(f'GPU available: {len(tf.config.list_physical_devices("GPU")) > 0}')

# Ensure results directory exists
RESULTS_DIR = "results"
os.makedirs(RESULTS_DIR, exist_ok=True)
print(f"Results will be saved to: {os.path.abspath(RESULTS_DIR)}")


## 1. Veri Yükleme ve Hazırlık

In [ ]:
PROCESSED_PATH = '../data/processed/features_engineered.csv'
DATA_PATH = '/kaggle/input/wind-turbine-gearbox-anomaly-detection-5year-scada/'

if os.path.exists(PROCESSED_PATH):
    df = pd.read_csv(PROCESSED_PATH, index_col=0, parse_dates=True)
else:
    csv_files = glob.glob(os.path.join(DATA_PATH, '*.csv'))
    dfs = [pd.read_csv(f) for f in sorted(csv_files)]
    if not csv_files:
        raise FileNotFoundError(
            f'No CSV files found in {DATA_PATH}. '
            'Run notebook 01 first or ensure the dataset is mounted.'
        )
    df = pd.concat(dfs, ignore_index=True)
    time_col = [c for c in df.columns if 'time' in c.lower() or 'date' in c.lower()]
    if time_col:
        df[time_col[0]] = pd.to_datetime(df[time_col[0]])
        df = df.sort_values(time_col[0]).set_index(time_col[0])

anomaly_col = [c for c in df.columns if 'anomal' in c.lower() or 'label' in c.lower() or 'fault' in c.lower() or 'alarm' in c.lower() or 'fail' in c.lower() or 'error' in c.lower() or 'status' in c.lower()]
ANOMALY_COL = anomaly_col[0] if anomaly_col else df.columns[-1]
# Ensure the anomaly column is binary (0/1);
# if values are continuous/multi-class, binarize: any non-zero → 1
_unique = df[ANOMALY_COL].dropna().unique()
if not set(_unique).issubset({0, 1, 0.0, 1.0, True, False}):
    print(f'Warning: {ANOMALY_COL!r} has non-binary values {sorted(_unique)[:5]}...'
          ' — binarizing (0=normal, >0=anomaly).')
    df[ANOMALY_COL] = (df[ANOMALY_COL] != 0).astype(int)

X_raw = df.select_dtypes(include=[np.number]).drop(columns=[ANOMALY_COL], errors='ignore')
y_raw = df[ANOMALY_COL].astype(int)
X_raw = X_raw.replace([np.inf, -np.inf], np.nan).fillna(0)

# Ölçeklendirme
scaler = StandardScaler()

# Temporal split
SPLIT_RATIO = 0.8
split_idx = int(len(X_raw) * SPLIT_RATIO)
val_idx = int(split_idx * 0.875)

X_arr = scaler.fit_transform(X_raw.values)
y_arr = y_raw.values

print(f'Data shape: {X_arr.shape}')
print(f'Anomaly rate: {y_arr.mean()*100:.2f}%')

## 2. Sliding Window Dataset

Zaman serisi modellerinde her örnek, geçmiş `window_size` adımını içerir.
- **window_size = 48**: Son 48 saati (2 gün) gören pencere
- **stride = 1**: Her adımda 1 zaman adımı ilerle
- **Label**: Penceredeki son adımın etiketi

In [ ]:
WINDOW_SIZE = 48
STRIDE = 1

def create_sliding_windows(X, y, window_size=48, stride=1):
    """Sliding window ile 3D veri seti oluştur."""
    X_windows, y_windows = [], []
    for i in range(0, len(X) - window_size, stride):
        X_windows.append(X[i:i + window_size])
        y_windows.append(y[i + window_size - 1])
    return np.array(X_windows, dtype=np.float32), np.array(y_windows, dtype=np.float32)

print('Creating sliding windows...')
X_windows, y_windows = create_sliding_windows(X_arr, y_arr, WINDOW_SIZE, STRIDE)
print(f'Windows shape: {X_windows.shape}  →  (N, window_size, n_features)')
print(f'Labels shape: {y_windows.shape}')

# Temporal split
n_windows = len(X_windows)
w_split = int(n_windows * SPLIT_RATIO)
w_val = int(w_split * 0.875)

X_tr = X_windows[:w_val]
y_tr = y_windows[:w_val]
X_val = X_windows[w_val:w_split]
y_val = y_windows[w_val:w_split]
X_te = X_windows[w_split:]
y_te = y_windows[w_split:]

print(f'\nTrain: {X_tr.shape} | Val: {X_val.shape} | Test: {X_te.shape}')
print(f'Train anomaly rate: {y_tr.mean()*100:.2f}%')
print(f'Test anomaly rate:  {y_te.mean()*100:.2f}%')

N_FEATURES = X_windows.shape[2]

## 3. LSTM Tabanlı Anomali Tespiti

LSTM (Long Short-Term Memory) ağları, uzun vadeli bağımlılıkları öğrenmek için cell state ve gate mekanizması kullanır.

In [ ]:
def build_lstm_model(window_size, n_features, n_units=128, dropout=0.3):
    """Stacked LSTM anomali sınıflandırıcısı."""
    inputs = keras.Input(shape=(window_size, n_features))
    
    x = layers.LSTM(n_units, return_sequences=True,
                    dropout=dropout, recurrent_dropout=0.1)(inputs)
    x = layers.LSTM(n_units // 2, return_sequences=False,
                    dropout=dropout)(x)
    x = layers.Dense(64, activation='relu')(x)
    x = layers.BatchNormalization()(x)
    x = layers.Dropout(dropout)(x)
    x = layers.Dense(32, activation='relu')(x)
    outputs = layers.Dense(1, activation='sigmoid')(x)
    
    model = keras.Model(inputs, outputs, name='LSTM_Classifier')
    return model

def get_callbacks(model_name, min_delta=1e-4, patience=10):
    """EarlyStopping + ReduceLROnPlateau callbacks."""
    return [
        keras.callbacks.EarlyStopping(
            monitor='val_loss', patience=patience,
            min_delta=min_delta, restore_best_weights=True
        ),
        keras.callbacks.ReduceLROnPlateau(
            monitor='val_loss', factor=0.5,
            patience=5, min_lr=1e-6, verbose=0
        ),
        keras.callbacks.ModelCheckpoint(
            f'results/{model_name}_best.h5',
            monitor='val_loss', save_best_only=True, verbose=0
        )
    ]

def compile_and_train(model, X_tr, y_tr, X_val, y_val, 
                      model_name, epochs=50, batch_size=256):
    """Modeli derle ve eğit."""
    # Class weights
    pos_weight = (y_tr == 0).sum() / (y_tr == 1).sum()
    class_weight = {0: 1.0, 1: float(pos_weight)}
    
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=1e-3),
        loss='binary_crossentropy',
        metrics=['accuracy', keras.metrics.AUC(name='auc')]
    )
    
    history = model.fit(
        X_tr, y_tr,
        validation_data=(X_val, y_val),
        epochs=epochs,
        batch_size=batch_size,
        class_weight=class_weight,
        callbacks=get_callbacks(model_name),
        verbose=1
    )
    return history

# LSTM eğitimi
lstm_model = build_lstm_model(WINDOW_SIZE, N_FEATURES)
lstm_model.summary()
lstm_history = compile_and_train(lstm_model, X_tr, y_tr, X_val, y_val, 'lstm')

## 4. TCN — Temporal Convolutional Network

TCN, dilated causal convolution kullanarak paralel hesaplama ile uzun vadeli bağımlılıkları öğrenir. LSTM'e göre daha hızlı eğitim süresi sağlar.

In [ ]:
def residual_block(x, filters, kernel_size, dilation_rate, dropout=0.2):
    """TCN için residual block with dilated causal convolution."""
    # Causal padding
    padding = (kernel_size - 1) * dilation_rate
    
    # Branch 1
    conv1 = layers.Conv1D(
        filters, kernel_size, padding='causal',
        dilation_rate=dilation_rate, activation='relu'
    )(x)
    conv1 = layers.SpatialDropout1D(dropout)(conv1)
    
    conv2 = layers.Conv1D(
        filters, kernel_size, padding='causal',
        dilation_rate=dilation_rate, activation='relu'
    )(conv1)
    conv2 = layers.SpatialDropout1D(dropout)(conv2)
    
    # Residual connection
    if x.shape[-1] != filters:
        x = layers.Conv1D(filters, 1)(x)
    
    return layers.Add()([x, conv2])

def build_tcn_model(window_size, n_features, filters=64, kernel_size=3, n_blocks=4):
    """Temporal Convolutional Network."""
    inputs = keras.Input(shape=(window_size, n_features))
    
    x = inputs
    for i in range(n_blocks):
        dilation = 2 ** i
        x = residual_block(x, filters, kernel_size, dilation)
    
    # Global pooling
    x = layers.GlobalAveragePooling1D()(x)
    x = layers.Dense(64, activation='relu')(x)
    x = layers.Dropout(0.3)(x)
    outputs = layers.Dense(1, activation='sigmoid')(x)
    
    model = keras.Model(inputs, outputs, name='TCN_Classifier')
    return model

tcn_model = build_tcn_model(WINDOW_SIZE, N_FEATURES)
tcn_model.summary()
tcn_history = compile_and_train(tcn_model, X_tr, y_tr, X_val, y_val, 'tcn')

## 5. Transformer Encoder (PatchTST Benzeri)

Transformer encoder, self-attention mekanizması ile zaman serisi içindeki uzun vadeli bağımlılıkları paralel olarak öğrenir. PatchTST yaklaşımı gibi, zaman serisi pencerelerini patch'lere böler.

In [ ]:
def positional_encoding(seq_len, d_model):
    """Sinüzoidal pozisyonel kodlama."""
    positions = np.arange(seq_len)[:, np.newaxis]
    dims = np.arange(d_model)[np.newaxis, :]
    angles = positions / np.power(10000, (2 * (dims // 2)) / d_model)
    angles[:, 0::2] = np.sin(angles[:, 0::2])
    angles[:, 1::2] = np.cos(angles[:, 1::2])
    return tf.cast(angles[np.newaxis, :, :], dtype=tf.float32)

class TransformerBlock(layers.Layer):
    """Tek Transformer encoder bloğu."""
    
    def __init__(self, d_model, n_heads, ff_dim, dropout=0.1, **kwargs):
        super().__init__(**kwargs)
        self.attention = layers.MultiHeadAttention(
            num_heads=n_heads, key_dim=d_model // n_heads
        )
        self.ffn = keras.Sequential([
            layers.Dense(ff_dim, activation='relu'),
            layers.Dense(d_model)
        ])
        self.norm1 = layers.LayerNormalization(epsilon=1e-6)
        self.norm2 = layers.LayerNormalization(epsilon=1e-6)
        self.dropout1 = layers.Dropout(dropout)
        self.dropout2 = layers.Dropout(dropout)
    
    def call(self, x, training=False):
        attn_output = self.attention(x, x, training=training)
        attn_output = self.dropout1(attn_output, training=training)
        x = self.norm1(x + attn_output)
        
        ffn_output = self.ffn(x)
        ffn_output = self.dropout2(ffn_output, training=training)
        return self.norm2(x + ffn_output)

def build_transformer_model(window_size, n_features, d_model=64, 
                            n_heads=4, ff_dim=128, n_blocks=2, dropout=0.1):
    """Transformer encoder sınıflandırıcısı."""
    inputs = keras.Input(shape=(window_size, n_features))
    
    # Feature projection
    x = layers.Dense(d_model)(inputs)
    
    # Positional encoding
    pos_enc = positional_encoding(window_size, d_model)
    x = x + pos_enc
    x = layers.Dropout(dropout)(x)
    
    # Transformer blocks
    for _ in range(n_blocks):
        x = TransformerBlock(d_model, n_heads, ff_dim, dropout)(x)
    
    # Classification head
    x = layers.GlobalAveragePooling1D()(x)
    x = layers.Dense(64, activation='relu')(x)
    x = layers.Dropout(dropout)(x)
    outputs = layers.Dense(1, activation='sigmoid')(x)
    
    model = keras.Model(inputs, outputs, name='Transformer_Classifier')
    return model

transformer_model = build_transformer_model(WINDOW_SIZE, N_FEATURES)
transformer_model.summary()
transformer_history = compile_and_train(
    transformer_model, X_tr, y_tr, X_val, y_val, 'transformer'
)

## 6. Eğitim/Validasyon Loss Grafikleri

In [ ]:
histories = {
    'LSTM': lstm_history,
    'TCN': tcn_history,
    'Transformer': transformer_history
}

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
colors = [('#e74c3c', '#c0392b'), ('#3498db', '#2980b9'), ('#2ecc71', '#27ae60')]

for (name, history), (c_train, c_val), ax in zip(histories.items(), colors, axes):
    epochs = range(1, len(history.history['loss']) + 1)
    ax.plot(epochs, history.history['loss'], color=c_train, linewidth=2, label='Train Loss')
    ax.plot(epochs, history.history['val_loss'], color=c_val, linewidth=2,
            linestyle='--', label='Val Loss')
    ax.set_title(f'{name} — Training History', fontsize=12)
    ax.set_xlabel('Epoch')
    ax.set_ylabel('Loss')
    ax.legend()
    ax.grid(True, alpha=0.3)

plt.suptitle('Training Histories', fontsize=14)
plt.tight_layout()
plt.savefig('results/training_histories.png', dpi=150, bbox_inches='tight')
plt.show()
# Eğitim geçmişlerini CSV olarak kaydet
for name, history in histories.items():
    hist_df = pd.DataFrame(history.history)
    hist_df.index.name = "epoch"
    fname = name.lower().replace(" ", "_")
    hist_df.to_csv(f"{RESULTS_DIR}/training_history_{fname}.csv")
print("Saved: training_history_*.csv")


## 7. Test Seti Değerlendirmesi ve Karşılaştırma

In [ ]:
dl_models = {
    'LSTM': lstm_model,
    'TCN': tcn_model,
    'Transformer': transformer_model
}

dl_results = []
dl_probas = {}

for name, model in dl_models.items():
    y_proba = model.predict(X_te, batch_size=512, verbose=0).flatten()
    y_pred = (y_proba >= 0.5).astype(int)
    
    metrics = {
        'Model': name,
        'Precision': precision_score(y_te, y_pred, zero_division=0),
        'Recall': recall_score(y_te, y_pred, zero_division=0),
        'F1': f1_score(y_te, y_pred, zero_division=0),
        'ROC-AUC': roc_auc_score(y_te, y_proba),
        'PR-AUC': average_precision_score(y_te, y_proba)
    }
    dl_results.append(metrics)
    dl_probas[name] = y_proba
    print(f'{name}: F1={metrics["F1"]:.4f} | ROC-AUC={metrics["ROC-AUC"]:.4f} | PR-AUC={metrics["PR-AUC"]:.4f}')

dl_results_df = pd.DataFrame(dl_results).set_index('Model')
print('\n=== DEEP LEARNING MODEL COMPARISON ===')
print(dl_results_df.round(4))

In [ ]:
# ROC ve PR karşılaştırması
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
colors = ['#e74c3c', '#3498db', '#2ecc71']

for (name, proba), color in zip(dl_probas.items(), colors):
    fpr, tpr, _ = roc_curve(y_te, proba)
    auc = roc_auc_score(y_te, proba)
    axes[0].plot(fpr, tpr, color=color, label=f'{name} ({auc:.3f})', linewidth=2)
    
    precision, recall, _ = precision_recall_curve(y_te, proba)
    ap = average_precision_score(y_te, proba)
    axes[1].plot(recall, precision, color=color, label=f'{name} ({ap:.3f})', linewidth=2)

axes[0].plot([0,1], [0,1], 'k--'); axes[0].set_title('ROC Curves')
axes[0].set_xlabel('FPR'); axes[0].set_ylabel('TPR')
axes[0].legend(); axes[0].grid(True, alpha=0.3)

axes[1].set_title('PR Curves')
axes[1].set_xlabel('Recall'); axes[1].set_ylabel('Precision')
axes[1].legend(); axes[1].grid(True, alpha=0.3)

plt.suptitle('Deep Learning Models — Test Set Performance', fontsize=14)
plt.tight_layout()
plt.savefig('results/dl_roc_pr_curves.png', dpi=150, bbox_inches='tight')
plt.show()

# Karşılaştırma ısı haritası
fig, ax = plt.subplots(figsize=(10, 4))
metric_cols = ['Precision', 'Recall', 'F1', 'ROC-AUC', 'PR-AUC']
sns.heatmap(dl_results_df[metric_cols], annot=True, fmt='.4f',
            cmap='YlOrRd', ax=ax, vmin=0, vmax=1)
ax.set_title('Deep Learning Model Comparison', fontsize=13)
plt.tight_layout()
plt.savefig('results/dl_model_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

print('\n✅ Time Series Deep Learning Complete!')
# DL model metriklerini kaydet
dl_results_df.to_csv(f"{RESULTS_DIR}/dl_model_metrics.csv")
print("Saved: dl_model_metrics.csv")

# Test tahminlerini kaydet
dl_preds_df = pd.DataFrame(dl_probas, index=range(len(y_te)))
dl_preds_df["y_true"] = y_te
dl_preds_df.to_csv(f"{RESULTS_DIR}/dl_test_predictions.csv", index=False)
print("Saved: dl_test_predictions.csv")


## Özet

| Model | Güçlü Yönler | Zayıf Yönler |
|-------|-------------|-------------|
| **LSTM** | Uzun vadeli bağımlılık, gradyan sorunlarını çözer | Yavaş eğitim, sıralı işlem |
| **TCN** | Paralel eğitim, dilated receptive field | Çok uzun serilerde sınırlı |
| **Transformer** | Global dikkat, paralel, state-of-the-art | Yüksek bellek kullanımı |

**Sonraki Adım:** `05_Hybrid_Ensemble` — tüm modelleri birleştir